In [4]:
import pandas as pd
import plotly.graph_objects as go

# Load data
df = pd.read_csv(
    "data/src/conflated_database___2026_03_31__11_14_35.xlsx - TextReading_Included.csv"
)

col = "Ultimate Output semantics"

# Split cells into lists; missing values become ["missing"]
def parse_semantics(x):
    if pd.isna(x) or str(x).strip() == "":
        return ["missing"]
    return [v.strip() for v in str(x).split(",") if v.strip()]

df["output_list"] = df[col].apply(parse_semantics)

# 1. Count unique values
counts = (
    df["output_list"]
    .explode()
    .value_counts()
)

print(counts)

# Optional: save counts
counts.reset_index().rename(
    columns={"index": "semantic_element", "count": "count"}
).to_csv("output_semantics_counts.csv", index=False)

# 2. Sankey: most frequent → less frequent sequence within each paper
ordered_elements = counts.index.tolist()
rank = {v: i for i, v in enumerate(ordered_elements)}

flows = []

for items in df["output_list"]:
    # sort elements in each row by global popularity
    items_sorted = sorted(set(items), key=lambda x: rank[x])

    # create transitions: most popular -> next -> next
    for a, b in zip(items_sorted[:-1], items_sorted[1:]):
        flows.append((a, b))

flows_df = (
    pd.DataFrame(flows, columns=["source", "target"])
    .value_counts()
    .reset_index(name="value")
)

# Build node ids
nodes = pd.Index(
    pd.concat([flows_df["source"], flows_df["target"]]).unique()
)

source_ids = flows_df["source"].map(nodes.get_loc)
target_ids = flows_df["target"].map(nodes.get_loc)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=nodes.tolist(),
        pad=20,
        thickness=18
    ),
    link=dict(
        source=source_ids,
        target=target_ids,
        value=flows_df["value"]
    )
)])

fig.update_layout(
    title="Ultimate Output Semantics: flows from more frequent to less frequent elements",
    font_size=11,
    width=1200,
    height=800
)

# fig.show()

# Optional exports
# fig.write_html("output_semantics_sankey.html")

output_list
space contour/function    63
entrance door             46
interior doors            43
walls/columns             38
windows                   23
missing                   16
Multiple Models            4
furniture                  2
stairs                     2
space adjacencies          1
Name: count, dtype: int64


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'link': {'source': {'bdata': 'AAECAwAEBAACAAU=', 'dtype': 'i1'},
                       'target': {'bdata': 'AQIDBAMFBgcEAgY=', 'dtype': 'i1'},
                       'value': {'bdata': 'KioeFgcCAQEBAQE=', 'dtype': 'i1'}},
              'node': {'label': [space contour/function, entrance door, interior
                                 doors, walls/columns, windows, furniture, stairs,
                                 space adjacencies],
                       'pad': 20,
                       'thickness': 18},
              'type': 'sankey'}],
    'layout': {'font': {'size': 11},
               'height': 800,
               'template': '...',
               'title': {'text': 'Ultimate Output Semantics: flows from more frequent to less frequent elements'},
               'width': 1200}
})